# Nonlinear Model Fitting: Confidence vs. Prediction Intervals

**Companion notebook to:** _Fitting Non-linear Models in Python: Confidence vs. Prediction Intervals_

**Inspired by:** [Fitting non-linear models](https://tonyladson.wordpress.com/2016/06/20/fitting-non-linear-models/) — Tony Ladson, 20 June 2016.

**A note on this one, unlike the others in this series:** the tool that fetches Ladson's gists for this post declined to reproduce his source code verbatim, citing the original research dataset it's built on. So this notebook is **not a line-for-line port** the way the POT, water balance, and line-graph notebooks are — it's an original Python demonstration of the *same general method* his post covers (fitting a nonlinear model and getting honest uncertainty on the fit), applied to a different, standard hydrological example (a stage-discharge rating curve) with clearly synthetic data, rather than reconstructing his dataset secondhand. Read his original post for his actual worked example.

The specific problem his post flags as the hard part: R's `nls()` doesn't give you confidence or prediction intervals out of the box the way `lm()` does for linear models — you need a separate package (he mentions `propagate::predictNLS`) or to do the propagation yourself. This notebook does the propagation explicitly, in Python, so the mechanics are visible rather than hidden in a library call.

In [1]:
import numpy as np
from scipy import optimize, stats
import matplotlib.pyplot as plt
print('scipy:', __import__('scipy').__version__)

scipy: 1.17.1


## 1. Synthetic data: a stage-discharge rating curve

A standard nonlinear hydrometric relationship: discharge as a power-law function of stage above some effective zero-flow level, `Q = C * (h - h0)^n`. This is textbook rating-curve form, not tied to any specific real gauge -- the data below is generated from a known "true" relationship plus realistic noise, purely to demonstrate the fitting method.

In [2]:
rng = np.random.default_rng(20260901)

true_C, true_h0, true_n = 45.0, 0.15, 1.85
h_obs = np.sort(rng.uniform(0.3, 3.5, 22))
Q_true = true_C * (h_obs - true_h0) ** true_n
Q_obs = Q_true * (1 + rng.normal(0, 0.06, size=h_obs.size))  # ~6% measurement noise

print(f'{len(h_obs)} synthetic gaugings, stage range [{h_obs.min():.2f}, {h_obs.max():.2f}] m')
print(f'True parameters: C={true_C}, h0={true_h0}, n={true_n}')

22 synthetic gaugings, stage range [0.32, 3.40] m
True parameters: C=45.0, h0=0.15, n=1.85


## 2. Fitting with scipy.optimize.curve_fit

`curve_fit` returns both the parameter estimates and their covariance matrix — the covariance is what makes proper interval estimation possible afterward.

In [3]:
def rating_curve(h, C, h0, n):
    return C * np.maximum(h - h0, 1e-6) ** n

popt, pcov = optimize.curve_fit(rating_curve, h_obs, Q_obs, p0=[1.0, 0.0, 2.0], maxfev=10000)
C_fit, h0_fit, n_fit = popt
perr = np.sqrt(np.diag(pcov))

print(f'Fitted: C={C_fit:.2f}+/-{perr[0]:.2f}   h0={h0_fit:.3f}+/-{perr[1]:.3f}   n={n_fit:.3f}+/-{perr[2]:.3f}')
print(f'(true:  C={true_C}          h0={true_h0}           n={true_n})')

# Individual parameters are NOT well-identified here -- check the correlation matrix
D = np.sqrt(np.diag(pcov))
corr = pcov / np.outer(D, D)
print()
print('Parameter correlation matrix (C, h0, n):')
print(np.round(corr, 3))

Fitted: C=66.38+/-18.98   h0=0.371+/-0.202   n=1.541+/-0.177
(true:  C=45.0          h0=0.15           n=1.85)

Parameter correlation matrix (C, h0, n):
[[ 1.     0.985 -0.986]
 [ 0.985  1.    -0.95 ]
 [-0.986 -0.95   1.   ]]


The individual parameters are noticeably off from their "true" values, and the correlation matrix shows why: C, h0 and n are highly correlated (|r| > 0.9 for every pair) — a well-known feature of power-law rating curves, where several different (C, h0, n) combinations can trace out nearly the same curve over the range the data actually covers. This is a real, common gotcha in nonlinear fitting generally, not a bug in this example. It's worth checking *whether the fitted curve itself is still useful* separately from whether the individual parameters recovered their "true" values — which is exactly what the next section does.

## 3. Confidence interval vs. prediction interval, via the delta method

- **Confidence interval**: uncertainty in the *fitted curve itself* — where the true mean relationship probably lies, given parameter uncertainty alone.
- **Prediction interval**: uncertainty in a *new individual observation* — necessarily wider, since it includes both parameter uncertainty and the residual scatter around the curve.

The delta method propagates parameter covariance through the model's own gradient (Jacobian) to get the variance of the fitted curve at any point; the prediction interval adds the residual variance on top.

In [4]:
resid = Q_obs - rating_curve(h_obs, *popt)
dof = len(h_obs) - len(popt)
resid_var = np.sum(resid**2) / dof

def rating_curve_grad(h, C, h0, n):
    base = np.maximum(h - h0, 1e-6)
    dC = base ** n
    dh0 = -C * n * base ** (n - 1)
    dn = C * base ** n * np.log(base)
    return np.stack([dC, dh0, dn], axis=-1)

h_grid = np.linspace(h_obs.min(), h_obs.max(), 100)
J = rating_curve_grad(h_grid, *popt)          # shape (100, 3)
Q_grid = rating_curve(h_grid, *popt)

var_mean = np.einsum('ij,jk,ik->i', J, pcov, J)   # J @ pcov @ J.T, diagonal only
se_mean = np.sqrt(np.maximum(var_mean, 0))
se_pred = np.sqrt(var_mean + resid_var)

t_crit = stats.t.ppf(0.975, dof)
ci_lo, ci_hi = Q_grid - t_crit * se_mean, Q_grid + t_crit * se_mean
pi_lo, pi_hi = Q_grid - t_crit * se_pred, Q_grid + t_crit * se_pred

idx = np.argmin(np.abs(h_grid - 2.0))
print(f'At h=2.0 m: fitted Q={Q_grid[idx]:.1f}')
print(f'  95% CI: ({ci_lo[idx]:.1f}, {ci_hi[idx]:.1f})  width={ci_hi[idx]-ci_lo[idx]:.1f}')
print(f'  95% PI: ({pi_lo[idx]:.1f}, {pi_hi[idx]:.1f})  width={pi_hi[idx]-pi_lo[idx]:.1f}')
assert (pi_hi[idx] - pi_lo[idx]) > (ci_hi[idx] - ci_lo[idx]), 'PI should always be wider than CI'
print('  Confirmed: PI is wider than CI, as it must be.')

At h=2.0 m: fitted Q=140.7
  95% CI: (133.0, 148.4)  width=15.4
  95% PI: (114.6, 166.7)  width=52.1
  Confirmed: PI is wider than CI, as it must be.


In [5]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.fill_between(h_grid, pi_lo, pi_hi, color='steelblue', alpha=0.15, label='95% prediction interval')
ax.fill_between(h_grid, ci_lo, ci_hi, color='steelblue', alpha=0.35, label='95% confidence interval')
ax.plot(h_grid, Q_grid, color='steelblue', lw=2, label='Fitted rating curve')
ax.scatter(h_obs, Q_obs, color='black', s=25, zorder=5, label='Synthetic gaugings')
ax.set_xlabel('Stage, h (m)')
ax.set_ylabel('Discharge, Q (m$^3$/s)')
ax.set_title('Nonlinear rating curve fit: confidence vs. prediction interval', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../../images/2026-09_nonlinear-rating-curve-ci-pi.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size ... with Axes>

## References

- Ladson, A.R. (2016). [Fitting non-linear models](https://tonyladson.wordpress.com/2016/06/20/fitting-non-linear-models/) (topic inspiration; this notebook's specific data and code are original, not a port of his).